Lab 7 Assignment KNN

Robert Armenta,  Swati Swati

Dataset Number 1: https://www.kaggle.com/datasets/sidhus/crab-age-prediction

Dataset Number 2: https://www.kaggle.com/datasets/uciml/glass



In [ ]:
# Import required libraries
# -------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder  # Imported, but not used in this version
from sklearn.model_selection import (
    train_test_split,   # Splits data into training and testing sets
    cross_val_score     # Performs cross-validation for model evaluation
)
from sklearn.preprocessing import StandardScaler  # Standardizes feature values
from sklearn.neighbors import KNeighborsClassifier  # KNN classifier
from sklearn.metrics import (
    accuracy_score,         # Measures overall accuracy
    classification_report,  # Gives precision, recall, and F1-score
    confusion_matrix        # Creates confusion matrix for predictions
)

# -------------------------
# Print title/header
# -------------------------
print("=" * 60)
print("Crab Age Group Classification")
print("=" * 60)
print()

# ---------------------------------------------------
# Set random seed so results are reproducible each run
# ---------------------------------------------------
np.random.seed(42)

# -------------------------
# Load the dataset
# -------------------------
# This lets you upload your CSV file directly into Colab

from google.colab import files
uploaded = files.upload()

df = pd.read_csv("CrabAgePrediction.csv")

# Preview first few rows of the dataset
df.head()

# ----------------------------------------------------
# Remove the 'Sex' column since it will not be used
# in this KNN model
# ----------------------------------------------------
df.drop('Sex', axis=1, inplace=True)

# Display updated dataframe
df



In [ ]:
# Dataset overview
# -------------------------
print("STEP 2: Dataset Overview")
print("-" * 40)

# Print basic dataset information
print(f"Total samples:       {df.shape[0]}")          # Number of rows
print(f"Number of features:  {df.shape[1] - 1}")      # Excluding Age target
print(f"Number of classes:   {df['Age'].nunique()}")  # Unique Age values
print()

# Show how many records belong to each age
print("Class distribution:")
print(df['Age'].value_counts().to_string())
print()

# Show the first 5 rows clearly
print("First 5 rows:")
print(df.head().to_string(index=False))
print()

# ----------------------------------------------------
# Function to convert exact age into broader age groups
# ----------------------------------------------------
def age_group(age):
    """
    Converts a crab's numerical age into a category.

    Young  = Age 7 or below
    Middle = Age 8 to 11
    Old    = Age 12 or above
    """
    if age <= 7:
        return 'Young'
    elif age <= 11:
        return 'Middle'
    else:
        return 'Old'

# Create a new target column using the age grouping function
df['Age_Group'] = df['Age'].apply(age_group)



In [ ]:
# Data exploration
# -------------------------
print("STEP 3: Exploring the Data")
print("-" * 40)

# Show average measurements for each age group
print("Average measurements per age group:")
print(df.groupby('Age_Group').mean().round(3))
print()

# Print all column names in the dataset
print("Columns in dataset:")
print(df.columns.tolist())
print()

# ----------------------------------------------------
# Pairplot to visualize relationships between features
# and see whether age groups separate well
# ----------------------------------------------------
plot_features = [
    'Length', 'Diameter', 'Height', 'Weight',
    'Shucked Weight', 'Viscera Weight', 'Shell Weight', 'Age'
]

sns.pairplot(
    df[plot_features],       # Use selected columns only
    hue='Age',               # Color points by Age
    palette='Set2',          # Color theme
    diag_kind='kde',         # Density plots on diagonal
    plot_kws={
        'alpha': 0.5,        # Make points semi-transparent
        'edgecolor': 'w'     # White border around points
    }
)

plt.suptitle('Crab Feature Relationships', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('crab_pairplot.png', dpi=150)
plt.show()

print("   -> Pair plot saved as 'crab_pairplot.png'")
print("   -> Look for features where the colors separate well.")
print("      Those features will likely be helpful for KNN.")
print()



In [ ]:
# Prepare data for modeling
# -------------------------
print("STEP 4: Preparing the Data")
print("-" * 40)

# Separate predictor variables (X) and target variable (y)
X = df.drop(['Age_Group', 'Age'], axis=1)   # Input features
y = df['Age_Group']                         # Target labels

# ----------------------------------------------------
# Split data into training and testing sets
# 80% for training, 20% for testing
# stratify=y keeps class proportions balanced
# ----------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Print the size of each set
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print()

# Check class balance after splitting
print("Class balance check:")
print(f"  Training: {dict(y_train.value_counts())}")
print(f"  Testing:  {dict(y_test.value_counts())}")
print("  (Should be roughly proportional)")
print()

# ----------------------------------------------------
# Scale the features because KNN is distance-based
# Important:
# - fit_transform() on training data
# - transform() only on testing data
# ----------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled with StandardScaler.")
print()



In [ ]:
# Train and evaluate KNN
# -------------------------
print("STEP 5: Training and Evaluating")
print("-" * 40)

# Create the KNN model with K = 7 neighbors
knn = KNeighborsClassifier(n_neighbors=7)

# Train the model on the scaled training data
knn.fit(X_train_scaled, y_train)

# Predict age groups for the test set
y_pred = knn.predict(X_test_scaled)

# Calculate overall model accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.2%}")
print()

# Print classification metrics for each class
print("Classification Report:")
print(classification_report(y_test, y_pred))

# -------------------------
# Confusion matrix
# -------------------------
labels = ['Young', 'Middle', 'Old']
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('Actual Label', fontsize=12)
plt.title('Confusion Matrix — Crab Age Groups', fontsize=14)
plt.tight_layout()
plt.savefig('crab_agegroup_confusion.png', dpi=150)
plt.show()

print("   -> Confusion matrix saved as 'crab_agegroup_confusion.png'")
print()



In [ ]:
# Cross-validation to find best K
# -------------------------
print("STEP 6: Finding the Best K with Cross-Validation")
print("-" * 40)

# Try different K values from 1 to 9
k_range = range(1, 10)

# Store average CV scores and standard deviations
cv_means = []
cv_stds = []

for k in k_range:
    # Create a KNN model for the current K
    model = KNeighborsClassifier(n_neighbors=k)

    # Run cross-validation on training data only
    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=2,                 # Number of folds
        scoring='accuracy'
    )

    # Save results
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

    # Print progress for selected K values
    if k % 5 == 0 or k == 1:
        print(f"   K={k:2d} → CV Accuracy = {scores.mean():.2%} (±{scores.std():.2%})")

print()

# -------------------------
# Plot cross-validation results
# -------------------------
plt.figure(figsize=(9, 5))

cv_means = np.array(cv_means)
cv_stds = np.array(cv_stds)
k_list = list(k_range)

# Plot average accuracy line
plt.plot(
    k_list, cv_means,
    marker='o',
    color='#27ae60',
    linewidth=2,
    markersize=4,
    label='Mean CV Accuracy'
)

# Plot shaded uncertainty band
plt.fill_between(
    k_list,
    cv_means - cv_stds,
    cv_means + cv_stds,
    alpha=0.2,
    color='#27ae60',
    label='±1 Std Dev'
)

plt.xlabel('K (Number of Neighbors)', fontsize=12)
plt.ylabel('Cross-Validation Accuracy', fontsize=12)
plt.title('Finding the Best K with Cross-Validation', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('abalone_best_k.png', dpi=150)
plt.show()

# Identify the best K
best_k = k_list[np.argmax(cv_means)]
best_cv_accuracy = cv_means.max()

print(f"   -> Best K = {best_k}")
print(f"   -> Best CV Accuracy = {best_cv_accuracy:.2%}")
print(f"   -> Chart saved as 'abalone_best_k.png'")
print()


In [ ]:
# Final evaluation with best K
# -------------------------
print("STEP 7: Final Evaluation with Best K")
print("-" * 40)

# Train final model using the best K value
final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(X_train_scaled, y_train)

# Predict on the test data
y_pred_final = final_model.predict(X_test_scaled)

# Calculate final test accuracy
final_accuracy = accuracy_score(y_test, y_pred_final)
print(f"Final Test Accuracy (K={best_k}): {final_accuracy:.2%}")
print()

# Final classification report
print("Final Classification Report:")
print(classification_report(y_test, y_pred_final))



In [ ]:
# Predict new crab samples
# -------------------------
print("STEP 8: Predicting New Crabs")
print("-" * 40)

# Create a dataframe with new crab measurements
new_crabs = pd.DataFrame({
    'Length': [0.90, 1.30, 1.55],
    'Diameter': [0.70, 1.00, 1.20],
    'Height': [0.22, 0.34, 0.41],
    'Weight': [6.0, 13.0, 19.5],
    'Shucked Weight': [2.5, 5.8, 8.4],
    'Viscera Weight': [1.2, 2.7, 4.0],
    'Shell Weight': [2.0, 6.0, 8.9],
})

print("New crab measurements:")
print(new_crabs.to_string(index=False))
print()

# ----------------------------------------------------
# Scale new data using the SAME scaler from training
# Never fit again on new data
# ----------------------------------------------------
new_scaled = scaler.transform(new_crabs)

# Use the trained KNN model to predict age groups
predictions = knn.predict(new_scaled)
probabilities = knn.predict_proba(new_scaled)

# Store class labels
class_labels = knn.classes_

# Print prediction results for each crab
for i in range(len(new_crabs)):
    print(f"Crab #{i+1}:")
    print(f"   Predicted age group: {predictions[i]}")
    print("   Confidence breakdown:")

    for cls, prob in zip(class_labels, probabilities[i]):
        bar_length = int(prob * 25)
        bar = '█' * bar_length + '░' * (25 - bar_length)
        print(f"      {cls:8s} {bar} {prob:6.1%}")
    print()

**KNN Lab Report**

Using the CrabAgePrediction dataset, a K-Nearest Neighbors (KNN) classification model was developed to group crabs into three age categories—Young, Middle, and Old—based on physical measurements such as length, diameter, height, and various weight attributes. The dataset was first cleaned by removing the non-essential “Sex” column and transforming the numerical age variable into broader, more interpretable categories. Exploratory analysis indicated that weight-related features, in particular, showed clear differences across age groups, making them useful predictors. The data was then split into training (80%) and testing (20%) sets with balanced class distributions, and all features were standardized to ensure the distance-based KNN algorithm performed accurately.

The initial model achieved approximately 70% accuracy, reflecting a solid but improvable ability to classify age groups. Evaluation using a confusion matrix showed that most misclassifications occurred between neighboring age groups, which is expected due to overlapping physical characteristics as crabs mature. To improve performance, cross-validation was used to test multiple values of K, allowing for selection of the optimal number of neighbors and resulting in a more reliable final model. When applied to new crab data, the model successfully generated predictions along with probability-based confidence levels. Overall, this analysis demonstrates that KNN can effectively use measurable biological features to estimate age group classifications, while emphasizing the importance of data preprocessing, feature scaling, and model tuning in producing accurate and meaningful results.

In [ ]:
# Dataset Number 2

In [ ]:
# Upload the dataset file from your computer
from google.colab import files
uploaded = files.upload()

# Import necessary libraries for data handling, visualization, and modeling
import pandas as pd              # for data manipulation
import numpy as np               # for numerical operations
import matplotlib.pyplot as plt  # for plotting graphs
import seaborn as sns            # for advanced visualizations

In [ ]:
# Load the dataset (works even if filename changes slightly)
df = pd.read_csv(list(uploaded.keys())[0])

# Display first 5 rows to verify dataset loaded correctly
df.head()

In [ ]:
# Print basic information about dataset
print("Shape:", df.shape)  # number of rows and columns

print("\nColumns:")
print(df.columns)  # list of column names

print("\nMissing values:")
print(df.isnull().sum())  # check for missing data

In [ ]:
# Create a bar chart showing how many samples of each glass type exist
plt.figure(figsize=(6,4))
df["Type"].value_counts().sort_index().plot(kind="bar")

# Add labels and title
plt.title("Glass Type Distribution")
plt.xlabel("Glass Type")
plt.ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Separate input features (X) and target variable (y)
# X contains all columns except 'Type'
X = df.drop("Type", axis=1)

# y is the column we want to predict
y = df["Type"]

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training (70%) and testing (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,       # 30% test data
    random_state=42,      # ensures reproducibility
    stratify=y            # keeps class distribution balanced
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Create KNN model with K = 5 neighbors
knn = KNeighborsClassifier(n_neighbors=5)

# Train the model using training data
knn.fit(X_train_scaled, y_train)

In [ ]:
# Predict glass types for test data
y_pred = knn.predict(X_test_scaled)

# Show first 10 predictions
print(y_pred[:10])

In [ ]:
from IPython.display import display

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose().round(3)

display(
    report_df.style
    .background_gradient(cmap='YlGnBu')
    .set_properties(**{
        'border': '1px solid black',
        'text-align': 'center',
        'padding': '6px'
    })
    .set_caption("Final Classification Report")
)

In [ ]:
# Get sorted class labels (used for axis labels)
labels = sorted(y.unique())

# Create figure with larger size for better visibility
plt.figure(figsize=(8,6))

# Create heatmap for confusion matrix
sns.heatmap(
    cm,                      # confusion matrix values
    annot=True,              # show numbers inside boxes
    fmt='d',                 # format as integers
    cmap='YlGnBu',           # color style (yellow-green-blue)
    xticklabels=labels,      # x-axis labels (predicted classes)
    yticklabels=labels,      # y-axis labels (actual classes)
    linewidths=2,            # thickness of grid lines
    linecolor='black',       # grid line color
    square=True,             # make cells square
    cbar=True                # show color bar legend
)

# Add title and axis labels
plt.title("Confusion Matrix with Class Labels", fontsize=14, weight='bold')
plt.xlabel("Predicted Class", fontsize=12)
plt.ylabel("Actual Class", fontsize=12)

# Improve layout spacing
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
# Test multiple values of K to find the best one
k_values = range(1, 16)
acc_scores = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred_k = model.predict(X_test_scaled)
    score = accuracy_score(y_test, pred_k)
    acc_scores.append(score)
    print("K =", k, "Accuracy =", round(score, 4))

In [ ]:
# Plot accuracy for each K value
plt.figure(figsize=(8,5))
plt.plot(k_values, acc_scores, marker='o')

plt.title("Accuracy vs K")
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Accuracy")

plt.grid(True)
plt.show()

# Find best K
best_k = list(k_values)[np.argmax(acc_scores)]
print("Best K:", best_k)

In [ ]:
# Import necessary libraries for KNN model and evaluation
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Import libraries for data handling and visualization
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display  # ensures styled table displays in Colab


# ============================
# Train Final KNN Model
# ============================

# Create KNN model using the best K value found earlier
best_knn = KNeighborsClassifier(n_neighbors=best_k)

# Train the model using the scaled training data
best_knn.fit(X_train_scaled, y_train)


# ============================
# Make Predictions
# ============================

# Predict glass types for the test dataset
best_pred = best_knn.predict(X_test_scaled)


# ============================
# Calculate Accuracy
# ============================

# Compute overall model accuracy
final_acc = accuracy_score(y_test, best_pred)

# Print formatted results header
print("=" * 55)
print("FINAL KNN MODEL RESULTS")
print("=" * 55)

# Display best K and final accuracy
print(f"Best K: {best_k}")
print(f"Final Accuracy: {final_acc:.4f}")

print("=" * 55)


# ============================
# Classification Report (Styled)
# ============================

# Generate classification report as a dictionary
report = classification_report(y_test, best_pred, output_dict=True)

# Convert report into a DataFrame for easier formatting
report_df = pd.DataFrame(report).transpose()

# Display styled table with colors for better readability
display(
    report_df.style
    .background_gradient(cmap="YlGnBu")  # color gradient based on values
    .set_caption("Final Classification Report")  # table title
    .set_properties(**{
        "border": "1px solid black",     # adds borders to cells
        "text-align": "center",          # center-align text
        "padding": "6px"                 # spacing inside cells
    })
)


# ============================
# Confusion Matrix
# ============================

# Create confusion matrix (actual vs predicted values)
cm = confusion_matrix(y_test, best_pred)

# Get sorted class labels for axis labeling
labels = sorted(y.unique())

# Create figure for confusion matrix
plt.figure(figsize=(8,6))

# Plot heatmap of confusion matrix
sns.heatmap(
    cm,                     # matrix values
    annot=True,             # show numbers inside boxes
    fmt="d",                # integer format
    cmap="YlGnBu",          # color scheme
    xticklabels=labels,     # predicted class labels
    yticklabels=labels,     # actual class labels
    linewidths=2,           # thickness of grid lines
    linecolor="black",      # grid line color
    square=True             # make cells square
)

# Add title and axis labels
plt.title("Final Confusion Matrix", fontsize=14, weight="bold")
plt.xlabel("Predicted Class", fontsize=12)
plt.ylabel("Actual Class", fontsize=12)

# Adjust layout for better spacing
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
# Allow user to input new glass sample values
print("Enter values for a new glass sample:")

RI = float(input("RI: "))
Na = float(input("Na: "))
Mg = float(input("Mg: "))
Al = float(input("Al: "))
Si = float(input("Si: "))
K  = float(input("K: "))
Ca = float(input("Ca: "))
Ba = float(input("Ba: "))
Fe = float(input("Fe: "))

# Create new data point
new_sample = pd.DataFrame([{
    "RI": RI,
    "Na": Na,
    "Mg": Mg,
    "Al": Al,
    "Si": Si,
    "K": K,
    "Ca": Ca,
    "Ba": Ba,
    "Fe": Fe
}])

# Scale new data using same scaler
new_sample_scaled = scaler.transform(new_sample)

# Predict glass type
prediction = best_knn.predict(new_sample_scaled)

print("Predicted Glass Type:", prediction[0])

**Final Report**

In this lab, the K-Nearest Neighbors (KNN) algorithm was used to examine the Glass dataset. In order to comprehend the structure of the dataset, it was first loaded and examined. This included looking for missing values and analyzing the distribution of glass kinds.

After then, the data was divided into training and testing sets. Since KNN is a distance-based method, StandardScaler was used to apply feature scaling so that all variables were on the same scale.

Accuracy, classification report, and confusion matrix were used to assess an initial KNN model that was trained using K=5. Various values of K were explored to enhance the model, and the optimal K value was chosen based on accuracy.

Using the ideal K value, a final model was constructed and reevaluated. Lastly, the model was utilized to forecast the kind of fresh glass sample that the user would enter.

This lab shows how KNN can be applied to classification problems and how adjusting parameters like the number of neighbors can enhance model performance.